# 11_train_combined — fingerprint + descriptor 결합 비교

**한 줄 요약:** fingerprint(1024)와 descriptor(217)를 **이어붙인 결합** 표현을 만들어, fingerprint만 / descriptor만 / 결합 셋을 같은 조건에서 비교한다(결합이 이득 있는지).
**큰 흐름:** ① 준비·읽기·정리 → ② 세 표현 만들고 각각 5-fold 비교

> **📌 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]**. ③은 그 셀에 **처음 나온** 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기
어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)*: `os.chdir('..')`=상위 폴더 이동, `print`=출력.

### 셀 1 — 도구 + 데이터 읽고 물질 단위 정리
라이브러리를 가져오고 descriptor CSV를 읽어 물질 단위로 합친다.

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier

SRC = "data/HSD17B13_descriptors.csv"   # canonical_smiles + label + descriptor 217
META = ["canonical_smiles", "ic50_nM", "relation", "sources", "label"]
NBITS = 1024

df = pd.read_csv(SRC).dropna(subset=["label"]).copy()
desc_cols = [c for c in df.columns if c not in META]
agg = {c: "first" for c in desc_cols}
agg["label"] = "max"
comp = df.groupby("canonical_smiles").agg(agg).reset_index()
comp["label"] = comp["label"].astype(int)

🔎 **코드 뜯어보기 (셀 1)** *(Pipeline·SimpleImputer는 10, groupby.agg는 10에서 설명)*
- descriptor CSV를 읽어 `label` 있는 행만 남기고 물질 단위로 정리(같은 코드 재사용).

### 셀 2 — 세 표현(FP/DESC/결합) 만들고 비교
각 분자의 fingerprint와 descriptor를 계산해 셋을 만들고, 같은 방식(5-fold CV)으로 채점해 결합이 나은지 본다.

In [ ]:
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NBITS)
fp_rows, desc_rows, y = [], [], []
D = comp[desc_cols].replace([np.inf, -np.inf], np.nan).to_numpy()
for i, smi in enumerate(comp["canonical_smiles"]):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        continue
    fp_rows.append(gen.GetFingerprintAsNumPy(mol))
    desc_rows.append(D[i])
    y.append(int(comp["label"].iloc[i]))
y = np.array(y)
FP = np.vstack(fp_rows).astype(np.float32)
DESC = np.vstack(desc_rows).astype(np.float32)
COMB = np.hstack([FP, DESC])
print(f"물질 {len(y)}개 | active {int(y.sum())} / inactive {int((y==0).sum())}")
print(f"특징 차원: FP {FP.shape[1]} | DESC {DESC.shape[1]} | 결합 {COMB.shape[1]}")


def evaluate(name, X):
    pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(n_estimators=400, class_weight="balanced",
                               random_state=42, n_jobs=-1, verbosity=-1)),
    ])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    proba = cross_val_predict(pipe, X, y, cv=skf, method="predict_proba", n_jobs=-1)[:, 1]
    pred = (proba >= 0.5).astype(int)
    tn, fp_, fn, tp = confusion_matrix(y, pred).ravel()
    auc = roc_auc_score(y, proba)
    pr = average_precision_score(y, proba)
    print(f"{name:16s} {auc:8.3f} {pr:8.3f} {tp/(tp+fn):12.3f} {tp/(tp+fp_):10.3f}")
    return auc


print("\n" + "=" * 62)
print(f"{'표현':16s} {'ROC-AUC':>8s} {'PR-AUC':>8s} {'Recall(act)':>12s} {'Prec(act)':>10s}")
print("=" * 62)
a1 = evaluate("fingerprint", FP)
a2 = evaluate("descriptor", DESC)
a3 = evaluate("결합(FP+DESC)", COMB)
print("=" * 62)
best = max([("fingerprint", a1), ("descriptor", a2), ("결합", a3)], key=lambda t: t[1])
print(f">>> 최고: {best[0]} (ROC-AUC {best[1]:.3f})")
gain = a3 - max(a1, a2)
print(f"결합이 단일 최고보다 {gain:+.3f} ROC-AUC "
      f"({'개선' if gain > 0.002 else '사실상 동일/미미'})")

🔎 **코드 뜯어보기 (셀 2)**
- `FP = np.vstack(fp_rows).astype(np.float32)` / `DESC = np.vstack(desc_rows)...` : fingerprint·descriptor를 각각 표로.
- `COMB = np.hstack([FP, DESC])` : **np.hstack**=두 표를 **좌우로**(가로) 이어붙이기 → 결합 표현.
- `def evaluate(name, X): ...` : 표현 하나를 받아 5-fold CV로 채점하고 ROC-AUC를 돌려주는 함수(내용은 05·10과 동일).
- `max([("fingerprint", a1), ...], key=lambda t: t[1])` : (이름, 점수) 쌍들 중 **점수(t[1])가 가장 큰 것**을 고르기. `lambda t: t[1]`=각 쌍의 2번째 값을 기준으로.
- `gain = a3 - max(a1, a2)` : 결합이 단일 최고보다 얼마나 나은지 차이.